**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-agentes](https://github.com/alansastre/genai-agentes)

---

# Tools

## Tools existentes

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

web_search = TavilySearch(max_results=3)

agent = create_agent("gpt-5.1", tools=[web_search])

inputs = {"messages": [{"role": "user", "content": "Indica un anuncio relevante de Google en el campo de la IA en la última semana"}]}

response = agent.invoke(inputs)
response

{'messages': [HumanMessage(content='Indica un anuncio relevante de Google en el campo de la IA en la última semana', additional_kwargs={}, response_metadata={}, id='231977dd-c0d6-4747-90b2-6f37936b49cc'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 1366, 'total_tokens': 1411, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-Cfks9j1J6zOLo24OP8SXLRI0UdjBg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--5488da3b-3d1d-48f1-bc84-f6a0a77a9d32-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'anuncio Google inteligencia artificial ultima semana', 'search_depth': 'basic', 'topic': 'new

In [ ]:
# el agente puede hacer varias llamadas a la Tool
inputs = {"messages": [{"role": "user", "content": "Indica el precio del Bitcoin hoy, y también el valor aproximado del VIX, dime los números."}]}

response = agent.invoke(inputs)
response

{'messages': [HumanMessage(content='Indica el precio del Bitcoin hoy, y también el valor del VIX.', additional_kwargs={}, response_metadata={}, id='afe8d928-907c-4bc5-a7d3-c30481a69b54'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 1365, 'total_tokens': 1449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1280}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-CfktXQhdkEliW0WRn1Q9r8dCJRHcM', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--7df5337c-85c8-4b50-91c3-4670195df7b7-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Bitcoin price today USD', 'search_depth': 'basic', 'topic': 'finance'}, 'id': 'call_X7yN5aDN7WajjRJthCsZdc09',

## Tools personalizadas

In [7]:
import requests
from langchain.tools import tool

@tool("get_coordinates") 
def get_coordinates(location):
    """ Obtener coordenadas a partir de lugar """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": location,
        "format": "json",
        "limit": 1
    }

    headers = {
        "User-Agent": "MiAplicacionPython/1.0 (contacto@ejemplo.com)"
    }

    respuesta = requests.get(url, params=params, headers=headers)
    datos = respuesta.json()

    if datos:
        lat = datos[0]["lat"]
        lon = datos[0]["lon"]
        return lat, lon
    else:
        return None

In [8]:
@tool("get_weather") 
def get_weather(latitude, longitude):
    """Herramienta para obtener la temperatura a partir de coordenadas"""
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [9]:
agent = create_agent("gpt-5.1", tools=[get_coordinates, get_weather])

inputs = {"messages": [{"role": "user", "content": "Indica qué temperatura dan hoy para El parque del retiro en Madrid España"}]}

response = agent.invoke(inputs)
response

{'messages': [HumanMessage(content='Indica qué temperatura dan hoy para El parque del retiro en Madrid España', additional_kwargs={}, response_metadata={}, id='0c31a86e-4999-460e-9b32-8bb28b6d2921'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 171, 'total_tokens': 202, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-Cfl6ANuKJAKYDDFOgpDicapp2bMg0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--4f37d3a1-3b6c-4d5b-9ac1-dcccea7f5e8b-0', tool_calls=[{'name': 'get_coordinates', 'args': {'location': 'Parque del Retiro, Madrid, España'}, 'id': 'call_8WRHxKCvxym00TiQYDicQOdV', 'type': 'tool_call'}],

## ToolRuntime

Lo construye automáticamente LangGraph por debajo utilizando los parámetros que hemos pasado en create_agent y invoke.

Permite a una tool acceder a toda la información: state, config, context, store...

* State - Mutable data that flows through execution (e.g., messages, counters, custom fields)
* Context - Immutable configuration like user IDs, session details, or application-specific configuration
* Store - Persistent long-term memory across conversations
* Stream Writer - Stream custom updates as tools execute
* Config - RunnableConfig for the execution
* Tool Call ID - ID of the current tool call

In [ ]:
from langchain.tools import tool, ToolRuntime

# Access the current conversation state
@tool
def summarize_conversation(
    runtime: ToolRuntime
) -> str:
    """Summarize the conversation so far."""
    messages = runtime.state["messages"]

    human_msgs = sum(1 for m in messages if m.__class__.__name__ == "HumanMessage")
    ai_msgs = sum(1 for m in messages if m.__class__.__name__ == "AIMessage")
    tool_msgs = sum(1 for m in messages if m.__class__.__name__ == "ToolMessage")

    return f"Conversation has {human_msgs} user messages, {ai_msgs} AI responses, and {tool_msgs} tool results"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime  # ToolRuntime parameter is not visible to the model
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")